# Datathon Task 1 V2 — Validation-Driven Model Search

V2 tidak mengasumsikan satu model pasti terbaik. Notebook ini menjalankan eksperimen terkontrol, memilih kandidat berdasarkan **out-of-time validation**, lalu membuat ensemble dan submission baru.

Pipeline:

1. Shared autoregressive Ridge sebagai anchor.
2. PCA city-state residual sebagai model spatial low-rank.
3. Empat neural ablation: temporal-only, directed graph, graph + gated text, dan graph + text + zero auxiliary task.
4. Arsitektur neural terbaik dilatih pada tiga seed dan semua training window.
5. Bobot AR/PCA/GNN dipilih terpisah untuk h5, h10, dan h15.
6. Semua model direfit pada full train sebelum membuat `/kaggle/working/submission_v2.csv`.

**Kaggle setup:** aktifkan GPU. Estimasi full run sekitar 10–20 menit. Untuk smoke test lokal gunakan environment variable `DATATON_FAST_MODE=1`.

## 1. Hipotesis dan success criteria

Temuan V1:

- AR validation MSE: **29.9994**.
- GraphSAGE GNN: **28.0709**.
- AR/GNN blend: **27.8935**.
- V1 hanya memakai setiap dua window dan prediksi antarhorizon masih sangat mirip.

Hipotesis V2:

- Directed incoming/outgoing aggregation lebih tepat daripada menggabungkan `A ∪ Aᵀ`.
- Road-specific text gating lebih informatif daripada broadcast text global.
- PCA residual memberikan error yang berbeda dari GNN dan dapat memperkuat ensemble.
- Multi-seed averaging mengurangi variance.
- Stride 1 memberi final model lebih banyak kondisi transisi.

**Success gate:** V2 dianggap improvement secara lokal hanya jika mean temporal-validation MSE lebih rendah dari 27.8935. Leaderboard test tetap menjadi penentu akhir.

In [ ]:
# Configuration and reproducibility
from __future__ import annotations

import gc
import json
import importlib
import os
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

BASE_SEED = 42
V1_REFERENCE_MSE = 27.8935
HORIZONS = (5, 10, 15)
N_ROADS = 1260
HISTORY = 15

KAGGLE_INPUT = Path("/kaggle/input/competitions/datathon-task-1/dataset-task1")
LOCAL_CANDIDATES = [Path.cwd() / "dataset-task1"] + [p / "dataset-task1" for p in Path.cwd().parents]
LOCAL_INPUT = next((p for p in LOCAL_CANDIDATES if p.exists()), LOCAL_CANDIDATES[0])
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT
IS_KAGGLE = KAGGLE_INPUT.exists()
FAST_MODE = os.getenv("DATATON_FAST_MODE", "0") == "1"
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else DATA_DIR.parent / "tmp/jupyter-notebook/v2-smoke"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_STEPS = 720 if not FAST_MODE else 120
EXPERIMENT_STRIDE = 2 if not FAST_MODE else 30
FINAL_STRIDE = 1 if not FAST_MODE else 30
PCA_DIM = 48 if not FAST_MODE else 8
TEXT_DIM = 32 if not FAST_MODE else 8
HIDDEN_DIM = 64 if not FAST_MODE else 16
EXPERIMENT_EPOCHS = 10 if not FAST_MODE else 1
SEED_SEARCH_EPOCHS = 15 if not FAST_MODE else 1
PATIENCE = 3
SEEDS = [42, 2026, 3407] if not FAST_MODE else [42]
BATCH_SIZE = 16 if torch.cuda.is_available() else (4 if not FAST_MODE else 2)
# Kaggle's PyTorch worker subprocess can lose core torch attributes during
# shared-memory collation. Keep loading in the stable main process.
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

# Kaggle's Python 3.12/PyTorch runtime can leave the internal submodule
# available on disk but unattached to the top-level torch package. AdamW
# imports torch._dynamo, which expects torch._utils to be attached.
if not hasattr(torch, "_utils"):
    torch._utils = importlib.import_module("torch._utils")

# Fail fast before expensive PCA/TF-IDF work if the optimizer runtime is broken.
_optimizer_probe_parameter = nn.Parameter(torch.tensor([1.0]))
_optimizer_probe = torch.optim.AdamW([_optimizer_probe_parameter], lr=1e-3)
_optimizer_probe_loss = (_optimizer_probe_parameter ** 2).sum()
_optimizer_probe_loss.backward()
_optimizer_probe.step()
del _optimizer_probe_parameter, _optimizer_probe, _optimizer_probe_loss
print("PyTorch optimizer preflight: OK | version", torch.__version__, "| torch._utils", hasattr(torch, "_utils"))

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(BASE_SEED)
print({
    "data": str(DATA_DIR), "output": str(OUTPUT_DIR), "device": str(DEVICE),
    "fast_mode": FAST_MODE, "experiment_stride": EXPERIMENT_STRIDE,
    "final_stride": FINAL_STRIDE, "seeds": SEEDS, "num_workers": NUM_WORKERS,
})
assert DATA_DIR.exists(), f"Dataset tidak ditemukan: {DATA_DIR}"


## 2. Load data dan sanity checks

Notebook mempertahankan ID order dari `sample_submission.csv`. Semua array diperiksa sebelum eksperimen dimulai agar model search tidak membuang waktu karena shape/order yang salah.

In [ ]:
def load_json_values(path: Path) -> list[str]:
    with path.open() as f:
        return list(json.load(f).values())

speed_paths = sorted((DATA_DIR / "train").glob("train_speed_*.npy"))
text_paths = sorted((DATA_DIR / "train").glob("train_text_*.json"))
train_speeds = [np.load(p, mmap_mode="r") for p in speed_paths]
train_texts = [load_json_values(p) for p in text_paths]
test_hist = np.load(DATA_DIR / "test/test_X_hist.npy")
test_texts = load_json_values(DATA_DIR / "test/test_texts.json")
adjacency = np.load(DATA_DIR / "static/matrix.npy")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert len(train_speeds) == len(train_texts) == 2
assert all(z.shape[1] == N_ROADS for z in train_speeds)
assert all(len(z) == len(t) for z, t in zip(train_speeds, train_texts))
assert test_hist.shape[1:] == (HISTORY, N_ROADS)
assert adjacency.shape == (N_ROADS, N_ROADS)
assert len(sample_submission) == len(test_hist) * 3 * N_ROADS

summary = pd.DataFrame([
    {"split": p.stem, "shape": str(z.shape), "mean": z.mean(), "std": z.std(), "zero_%": 100*np.mean(z==0)}
    for p, z in zip(speed_paths, train_speeds)
] + [{"split": "test", "shape": str(test_hist.shape), "mean": test_hist.mean(),
      "std": test_hist.std(), "zero_%": 100*np.mean(test_hist==0)}])
display(summary.round(3))
print("Directed edges including diagonal:", int(adjacency.sum()))


## 3. Temporal validation protocol

Dua hari terakhir dari masing-masing continuous block menjadi validation. Training target tidak pernah melewati cut point. Random split dilarang karena window yang berdekatan sangat overlap.

Ada dua training index:

- `experiment_times`: stride 2 untuk membandingkan banyak arsitektur dengan cepat.
- `final_train_times`: stride 1 untuk multi-seed model yang sudah terpilih.

Fast mode memangkas keduanya hanya untuk pengujian kode lokal.

In [ ]:
def valid_times(n, start, stop, stride=1):
    lo = max(HISTORY - 1, start)
    hi = min(stop, n - max(HORIZONS))
    return np.arange(lo, hi, stride, dtype=np.int32)

cuts = [len(z) - VAL_STEPS for z in train_speeds]
experiment_times = [valid_times(len(z), HISTORY-1, cut, EXPERIMENT_STRIDE) for z, cut in zip(train_speeds, cuts)]
final_train_times = [valid_times(len(z), HISTORY-1, cut, FINAL_STRIDE) for z, cut in zip(train_speeds, cuts)]
val_times = [valid_times(len(z), cut, len(z), 1) for z, cut in zip(train_speeds, cuts)]

if FAST_MODE:
    experiment_times = [t[-min(100, len(t)):] for t in experiment_times]
    final_train_times = [t[-min(120, len(t)):] for t in final_train_times]
    val_times = [t[-min(30, len(t)):] for t in val_times]

def collect_histories_targets(times_by_block):
    hist, targets = [], []
    for z, times in zip(train_speeds, times_by_block):
        for t in times:
            hist.append(np.asarray(z[t-HISTORY+1:t+1], dtype=np.float32))
            targets.append(np.stack([z[t+h] for h in HORIZONS], axis=-1).astype(np.float32))
    return np.stack(hist), np.stack(targets)

val_hist, val_y = collect_histories_targets(val_times)
print("Experiment train:", [len(x) for x in experiment_times])
print("Final stride-1 train:", [len(x) for x in final_train_times])
print("Validation:", [len(x) for x in val_times], val_hist.shape, val_y.shape)

def horizon_mse(y, pred):
    return np.array([np.mean((y[..., i]-pred[..., i])**2) for i in range(3)], dtype=np.float64)


## 4. Model A — shared autoregressive Ridge

AR tetap menjadi anchor karena V1 menunjukkan MSE sekitar 30 dan error-nya melengkapi GNN. Model memprediksi perubahan dari latest speed menggunakan 15 lag yang dicenter pada latest reading.

In [ ]:
def fit_shared_ar(times_by_block, max_timestamps_per_block=1400, alpha=1000.0, seed=BASE_SEED):
    rng = np.random.default_rng(seed)
    x_parts, y_parts = [], {h: [] for h in HORIZONS}
    for z, times in zip(train_speeds, times_by_block):
        take = min(len(times), max_timestamps_per_block if not FAST_MODE else 50)
        chosen = np.sort(rng.choice(times, size=take, replace=False))
        w = np.stack([z[t-HISTORY+1:t+1] for t in chosen]).astype(np.float32)
        x_parts.append((w-w[:,-1:]).transpose(0,2,1).reshape(-1,HISTORY))
        for h in HORIZONS:
            y_parts[h].append((z[chosen+h]-z[chosen]).reshape(-1).astype(np.float32))
    X = np.concatenate(x_parts)
    return {h: Ridge(alpha=alpha).fit(X, np.concatenate(y_parts[h])) for h in HORIZONS}

def predict_ar_histories(models, histories, batch=256):
    outputs = []
    for start in range(0, len(histories), batch):
        w = np.asarray(histories[start:start+batch], dtype=np.float32)
        x = (w-w[:,-1:]).transpose(0,2,1).reshape(-1,HISTORY)
        latest = w[:,-1].reshape(-1)
        outputs.append(np.stack([
            (latest+models[h].predict(x)).reshape(len(w),N_ROADS) for h in HORIZONS
        ], axis=-1))
    return np.concatenate(outputs).astype(np.float32)

ar_val_models = fit_shared_ar(experiment_times)
val_pred_ar = predict_ar_histories(ar_val_models, val_hist)
ar_mse = horizon_mse(val_y, val_pred_ar)
print("AR MSE per horizon / mean:", np.round(ar_mse,4), round(ar_mse.mean(),4))


## 5. Model B — PCA city-state residual

AR hanya melihat satu road. PCA residual melihat state seluruh kota melalui lima ringkasan temporal, lalu memprediksi koreksi multi-output untuk semua road. Alpha Ridge dipilih pada validation untuk setiap horizon.

Model ini sengaja linear dan low-rank: tujuannya bukan menggantikan GNN, melainkan memberi pola error berbeda untuk ensemble.

In [ ]:
def fit_road_pca(row_stops, n_components=PCA_DIM):
    rows = np.concatenate([np.asarray(z[:stop], dtype=np.float32) for z, stop in zip(train_speeds, row_stops)])
    model = PCA(n_components=n_components, whiten=True, svd_solver="randomized", random_state=BASE_SEED)
    model.fit(rows)
    del rows
    gc.collect()
    return model

def summarize_continuous(z, times):
    return np.stack([
        np.stack([z[t-14:t-11].mean(0), z[t-9:t-6].mean(0), z[t-4:t-1].mean(0),
                  z[t-2:t+1].mean(0), z[t]-z[t-4]], axis=0)
        for t in times
    ]).astype(np.float32)

def pca_features_for_times(pca, times_by_block, batch=256):
    out = []
    for z, times in zip(train_speeds, times_by_block):
        block = []
        for start in range(0, len(times), batch):
            s = summarize_continuous(z, times[start:start+batch])
            block.append(np.concatenate([pca.transform(s[:,k]) for k in range(5)], axis=1))
        out.append(np.concatenate(block))
    return np.concatenate(out).astype(np.float32)

def pca_features_for_histories(pca, histories):
    # histories: sample,time,road
    s = np.stack([
        histories[:,0:3].mean(1), histories[:,5:8].mean(1), histories[:,10:13].mean(1),
        histories[:,-3:].mean(1), histories[:,-1]-histories[:,-5]
    ], axis=1).astype(np.float32)
    return np.concatenate([pca.transform(s[:,k]) for k in range(5)], axis=1).astype(np.float32)

def targets_and_ar_for_times(times_by_block, ar_models, batch=128):
    targets, bases = [], []
    for z, times in zip(train_speeds, times_by_block):
        for start in range(0, len(times), batch):
            ts = times[start:start+batch]
            h = np.stack([z[t-HISTORY+1:t+1] for t in ts]).astype(np.float32)
            targets.append(np.stack([z[ts+hz] for hz in HORIZONS], axis=-1).astype(np.float32))
            bases.append(predict_ar_histories(ar_models, h, batch=batch))
    return np.concatenate(targets), np.concatenate(bases)

pca_val = fit_road_pca(cuts)
F_pca_train = pca_features_for_times(pca_val, experiment_times)
F_pca_val = pca_features_for_histories(pca_val, val_hist)
Y_pca_train, B_pca_train = targets_and_ar_for_times(experiment_times, ar_val_models)

pca_alphas = [1e2, 1e4, 1e5] if not FAST_MODE else [1e4]
pca_models, pca_best_alpha, pca_val_parts = {}, {}, []
for i, h in enumerate(HORIZONS):
    best = (float("inf"), None, None, None)
    residual = Y_pca_train[...,i] - B_pca_train[...,i]
    for alpha in pca_alphas:
        model = Ridge(alpha=alpha).fit(F_pca_train, residual)
        pred = val_pred_ar[...,i] + model.predict(F_pca_val)
        score = float(np.mean((val_y[...,i]-pred)**2))
        if score < best[0]: best = (score, alpha, model, pred.astype(np.float32))
    score, alpha, model, pred = best
    pca_models[h], pca_best_alpha[h] = model, alpha
    pca_val_parts.append(pred)
val_pred_pca = np.stack(pca_val_parts, axis=-1)
pca_mse = horizon_mse(val_y, val_pred_pca)
print("PCA alpha:", pca_best_alpha)
print("PCA residual MSE per horizon / mean:", np.round(pca_mse,4), round(pca_mse.mean(),4))
del Y_pca_train, B_pca_train
gc.collect()


## 6. Text preprocessing

TF-IDF n-gram mempertahankan tipe event dan lokasi. Truncated SVD membuat vektor compact. Pada model `graph_text_*`, vektor ini tidak hanya dibroadcast: learned road-event embedding membentuk gate berbeda untuk setiap road.

In [ ]:
all_train_text = [s for block in train_texts for s in block]
tfidf = TfidfVectorizer(ngram_range=(1,3), min_df=3, max_features=7000, sublinear_tf=True, dtype=np.float32)
text_sparse = tfidf.fit_transform(all_train_text)
text_dim_actual = min(TEXT_DIM, text_sparse.shape[1]-1)
text_svd = TruncatedSVD(n_components=text_dim_actual, n_iter=7, random_state=BASE_SEED)
all_text_z = text_svd.fit_transform(text_sparse).astype(np.float32)
test_text_z = text_svd.transform(tfidf.transform(test_texts)).astype(np.float32)
offsets = np.cumsum([0]+[len(x) for x in train_texts])
train_text_z = [all_text_z[offsets[i]:offsets[i+1]] for i in range(len(train_texts))]
print("TF-IDF / latent:", text_sparse.shape, all_text_z.shape,
      "explained", round(float(text_svd.explained_variance_ratio_.sum()),4))
del text_sparse, all_text_z
gc.collect()


## 7. Neural experiment family

Empat ablation memakai encoder dan optimization yang sama. Hanya signal path yang berbeda, sehingga perbandingan lebih adil.

| Experiment | Directed graph | Gated text | Zero auxiliary |
|---|---:|---:|---:|
| `temporal_only` | No | No | No |
| `directed_graph` | Yes | No | No |
| `graph_text_gate` | Yes | Yes | No |
| `graph_text_zero_aux` | Yes | Yes | Yes |

Directed block mempelajari incoming dan outgoing aggregation secara terpisah. Zero auxiliary head hanya memengaruhi representation learning; output speed tetap dioptimasi langsung dengan MSE.

In [ ]:
# Directed edge lists, excluding diagonal because self state has its own projection.
src_np, dst_np = np.nonzero(adjacency != 0)
keep = src_np != dst_np
src_np, dst_np = src_np[keep], dst_np[keep]
out_degree_np = np.bincount(src_np, minlength=N_ROADS).clip(min=1).astype(np.float32)
in_degree_np = np.bincount(dst_np, minlength=N_ROADS).clip(min=1).astype(np.float32)

norm_rows = np.concatenate([np.asarray(z[:cut], dtype=np.float32) for z, cut in zip(train_speeds, cuts)])
road_mean = norm_rows.mean(0).astype(np.float32)
road_std = (norm_rows.std(0)+3.0).astype(np.float32)
del norm_rows
gc.collect()

class TrafficDataset(Dataset):
    def __init__(self, times_by_block):
        self.index = [(b,int(t)) for b,times in enumerate(times_by_block) for t in times]
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        b,t = self.index[i]; z=train_speeds[b]
        hist=np.asarray(z[t-HISTORY+1:t+1],dtype=np.float32).T.copy()
        target=np.stack([z[t+h] for h in HORIZONS],axis=-1).astype(np.float32)
        text=np.asarray(train_text_z[b][t],dtype=np.float32)
        return torch.from_numpy(hist),torch.from_numpy(text),torch.from_numpy(target)

class DirectedGraphBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.self_linear=nn.Linear(hidden,hidden)
        self.in_linear=nn.Linear(hidden,hidden,bias=False)
        self.out_linear=nn.Linear(hidden,hidden,bias=False)
        self.norm=nn.LayerNorm(hidden); self.dropout=nn.Dropout(.10)
    def forward(self,h,src,dst,in_degree,out_degree):
        incoming=torch.zeros_like(h); outgoing=torch.zeros_like(h)
        incoming.index_add_(1,dst,h[:,src,:])
        outgoing.index_add_(1,src,h[:,dst,:])
        incoming=incoming/in_degree.view(1,-1,1)
        outgoing=outgoing/out_degree.view(1,-1,1)
        u=torch.nn.functional.gelu(self.self_linear(h)+self.in_linear(incoming)+self.out_linear(outgoing))
        return self.norm(h+self.dropout(u))

@dataclass(frozen=True)
class ModelConfig:
    name: str
    use_graph: bool
    use_text: bool
    zero_aux_weight: float = 0.0

class TrafficNetV2(nn.Module):
    def __init__(self, config, mean_np, std_np, hidden=HIDDEN_DIM):
        super().__init__(); self.config=config
        self.register_buffer("road_mean",torch.tensor(mean_np).view(1,N_ROADS,1))
        self.register_buffer("road_std",torch.tensor(std_np).view(1,N_ROADS,1))
        self.register_buffer("src",torch.tensor(src_np,dtype=torch.long))
        self.register_buffer("dst",torch.tensor(dst_np,dtype=torch.long))
        self.register_buffer("in_degree",torch.tensor(in_degree_np))
        self.register_buffer("out_degree",torch.tensor(out_degree_np))
        self.road_embedding=nn.Embedding(N_ROADS,12)
        self.history_encoder=nn.Sequential(
            nn.Linear(HISTORY*3+12,hidden),nn.GELU(),nn.Dropout(.10),
            nn.Linear(hidden,hidden),nn.GELU())
        if config.use_text:
            gate_dim=16
            self.text_context=nn.Sequential(nn.Linear(text_dim_actual,hidden),nn.GELU(),nn.Linear(hidden,hidden))
            self.text_query=nn.Linear(text_dim_actual,gate_dim)
            self.road_event_embedding=nn.Embedding(N_ROADS,gate_dim)
        self.graph_blocks=nn.ModuleList([DirectedGraphBlock(hidden),DirectedGraphBlock(hidden)]) if config.use_graph else nn.ModuleList()
        self.horizon_heads=nn.ModuleList([
            nn.Sequential(nn.Linear(hidden,hidden//2),nn.GELU(),nn.Linear(hidden//2,1)) for _ in HORIZONS
        ])
        self.zero_head=nn.Linear(hidden,len(HORIZONS))
        for head in self.horizon_heads:
            nn.init.zeros_(head[-1].weight); nn.init.zeros_(head[-1].bias)
    def forward(self,hist,text):
        normalized=(hist-self.road_mean)/self.road_std
        base=hist[:,:,-3:].mean(-1,keepdim=True)
        deviations=(hist-base)/self.road_std
        zero=(hist==0).to(hist.dtype)
        ids=torch.arange(N_ROADS,device=hist.device)
        emb=self.road_embedding(ids).unsqueeze(0).expand(len(hist),-1,-1)
        h=self.history_encoder(torch.cat([normalized,deviations,zero,emb],-1))
        if self.config.use_text:
            context=self.text_context(text).unsqueeze(1)
            query=self.text_query(text)
            gate=torch.sigmoid(torch.einsum('bd,rd->br',query,self.road_event_embedding.weight)/(query.shape[-1]**.5))
            h=h+gate.unsqueeze(-1)*context
        for block in self.graph_blocks:
            h=block(h,self.src,self.dst,self.in_degree,self.out_degree)
        residual=torch.cat([head(h) for head in self.horizon_heads],-1)*self.road_std
        return base+residual,self.zero_head(h)

ABLATIONS=[
    ModelConfig("temporal_only",False,False,0.0),
    ModelConfig("directed_graph",True,False,0.0),
    ModelConfig("graph_text_gate",True,True,0.0),
    ModelConfig("graph_text_zero_aux",True,True,0.25),
]
display(pd.DataFrame([asdict(x) for x in ABLATIONS]))


## 8. Training utilities dan ablation search

Early stopping memakai overall MSE. Tabel hasil tetap melaporkan ketiga horizon. Jika zero auxiliary aktif, weighted BCE ditambahkan ke training loss tetapi tidak ke validation metric.

In [ ]:
def make_loader(times, shuffle, seed, batch_size=BATCH_SIZE):
    generator=torch.Generator(); generator.manual_seed(seed)
    return DataLoader(TrafficDataset(times),batch_size=batch_size,shuffle=shuffle,
                      num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),
                      generator=generator,drop_last=False)

def evaluate_neural(model,loader,return_pred=False):
    model.eval(); sse=np.zeros(3,np.float64); n=0; parts=[]
    with torch.no_grad():
        for hist,text,target in loader:
            hist,text,target=hist.to(DEVICE),text.to(DEVICE),target.to(DEVICE)
            pred,_=model(hist,text)
            sse+=((pred-target)**2).sum((0,1)).cpu().numpy(); n+=target.shape[0]*target.shape[1]
            if return_pred: parts.append(pred.cpu().numpy())
    scores=sse/n
    return (scores,np.concatenate(parts)) if return_pred else scores

def train_neural(config,train_times_local,val_times_local,mean_np,std_np,seed,epochs,verbose=True):
    set_seed(seed)
    train_loader=make_loader(train_times_local,True,seed)
    val_loader=make_loader(val_times_local,False,seed) if val_times_local is not None else None
    model=TrafficNetV2(config,mean_np,std_np).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,epochs))
    best_state=None; best=float('inf'); best_epoch=0; stale=0; log=[]
    pos_weight=torch.tensor(5.0,device=DEVICE)
    for epoch in range(1,epochs+1):
        started=time.time(); model.train(); train_sse=0.; train_n=0
        for hist,text,target in train_loader:
            hist,text,target=hist.to(DEVICE),text.to(DEVICE),target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred,zero_logits=model(hist,text)
            mse_loss=torch.mean((pred-target)**2)
            loss=mse_loss
            if config.zero_aux_weight>0:
                zero_target=(target==0).to(target.dtype)
                aux=torch.nn.functional.binary_cross_entropy_with_logits(zero_logits,zero_target,pos_weight=pos_weight)
                loss=loss+config.zero_aux_weight*aux
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
            train_sse+=mse_loss.item()*target.numel(); train_n+=target.numel()
        scheduler.step(); train_mse=train_sse/train_n
        val_scores=evaluate_neural(model,val_loader) if val_loader is not None else np.array([train_mse]*3)
        val_mean=float(val_scores.mean())
        log.append({"epoch":epoch,"train_mse":train_mse,"val_mse":val_mean})
        if verbose: print(f"{config.name} seed={seed} epoch={epoch:02d} train={train_mse:.4f} val={val_mean:.4f} time={time.time()-started:.1f}s")
        if val_mean<best-1e-4:
            best=val_mean; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if val_loader is not None and stale>=PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model,pd.DataFrame(log),best_epoch,best

val_loader_fixed=make_loader(val_times,False,BASE_SEED)
ablation_rows=[]; ablation_predictions={}; ablation_epochs={}
for config in ABLATIONS:
    print("\n===",config.name,"===")
    model,log,best_epoch,best=train_neural(
        config,experiment_times,val_times,road_mean,road_std,BASE_SEED,EXPERIMENT_EPOCHS)
    scores,pred=evaluate_neural(model,val_loader_fixed,True)
    ablation_predictions[config.name]=pred.astype(np.float32)
    ablation_epochs[config.name]=best_epoch
    ablation_rows.append({"model":config.name,"h5":scores[0],"h10":scores[1],"h15":scores[2],
                          "mean":scores.mean(),"best_epoch":best_epoch})
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

ablation_report=pd.DataFrame(ablation_rows).sort_values("mean").reset_index(drop=True)
display(ablation_report.round(5))
best_name=ablation_report.iloc[0]["model"]
best_config=next(x for x in ABLATIONS if x.name==best_name)
print("Selected architecture:",best_config)


## 9. Stride-1 multi-seed confirmation

Ablation hanya memilih arsitektur. Kandidat terpilih kemudian dilatih ulang pada semua window train (`stride=1`) dengan tiga seed. Validation prediction dirata-ratakan sebelum ensemble fitting.

Jika satu seed jauh lebih buruk, hasilnya terlihat di tabel dan dapat dikeluarkan secara manual sebelum submission final.

In [ ]:
seed_val_predictions=[]; seed_rows=[]; seed_best_epochs=[]
for seed in SEEDS:
    print("\n=== CONFIRM SEED",seed,"===")
    model,log,best_epoch,best=train_neural(
        best_config,final_train_times,val_times,road_mean,road_std,seed,SEED_SEARCH_EPOCHS)
    scores,pred=evaluate_neural(model,val_loader_fixed,True)
    seed_val_predictions.append(pred.astype(np.float32)); seed_best_epochs.append(best_epoch)
    seed_rows.append({"seed":seed,"h5":scores[0],"h10":scores[1],"h15":scores[2],
                      "mean":scores.mean(),"best_epoch":best_epoch})
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

seed_report=pd.DataFrame(seed_rows)
display(seed_report.round(5))
val_pred_gnn=np.mean(seed_val_predictions,axis=0).astype(np.float32)
gnn_seed_mse=horizon_mse(val_y,val_pred_gnn)
refit_epochs=max(1,int(np.median(seed_best_epochs)))
print("Seed-averaged MSE:",np.round(gnn_seed_mse,5),"mean",round(gnn_seed_mse.mean(),5))
print("Full refit epochs (median):",refit_epochs)


## 10. Non-negative ensemble optimization

Untuk setiap horizon, optimizer memilih bobot AR, PCA, dan seed-averaged GNN dengan syarat:

- semua bobot ≥ 0;
- jumlah bobot = 1.

Optimization memakai Gram matrix tiga model, sehingga tidak perlu berulang kali memindai jutaan prediksi. Jika suatu model tidak membantu, bobotnya dapat menjadi nol.

In [ ]:
model_names=["AR","PCA","GNN"]
val_candidates=[val_pred_ar,val_pred_pca,val_pred_gnn]

def simplex_weights(y,predictions):
    X=np.stack([p.reshape(-1).astype(np.float64) for p in predictions],axis=1)
    y=y.reshape(-1).astype(np.float64)
    n=len(y); G=(X.T@X)/n; c=(X.T@y)/n
    fun=lambda w: float(w@G@w-2*c@w)
    jac=lambda w: 2*G@w-2*c
    result=minimize(fun,np.full(len(predictions),1/len(predictions)),jac=jac,
                    bounds=[(0,1)]*len(predictions),constraints={"type":"eq","fun":lambda w:w.sum()-1},
                    method="SLSQP",options={"ftol":1e-12,"maxiter":200})
    assert result.success,result.message
    return result.x

blend_weights=[]; blend_parts=[]; blend_rows=[]
for i,h in enumerate(HORIZONS):
    w=simplex_weights(val_y[...,i],[p[...,i] for p in val_candidates])
    pred=sum(w[j]*val_candidates[j][...,i] for j in range(len(w)))
    blend_weights.append(w); blend_parts.append(pred.astype(np.float32))
    blend_rows.append({"horizon":f"h{h}",**{f"w_{n}":w[j] for j,n in enumerate(model_names)},
                       "AR_MSE":ar_mse[i],"PCA_MSE":pca_mse[i],"GNN_MSE":gnn_seed_mse[i],
                       "blend_MSE":np.mean((val_y[...,i]-pred)**2)})
val_pred_blend=np.stack(blend_parts,axis=-1)
blend_report=pd.DataFrame(blend_rows)
display(blend_report.round(6))
v2_val_mse=float(blend_report.blend_MSE.mean())
print("V2 validation mean:",round(v2_val_mse,6))
if FAST_MODE:
    print("FAST_MODE score hanya smoke subset dan TIDAK comparable dengan V1 reference.")
else:
    print("V1 reference:",V1_REFERENCE_MSE)
    print("Delta vs V1:",round(v2_val_mse-V1_REFERENCE_MSE,6),
          "| relative improvement %:",round(100*(V1_REFERENCE_MSE-v2_val_mse)/V1_REFERENCE_MSE,3))

pd.concat([
    pd.DataFrame([{"model":"AR","h5":ar_mse[0],"h10":ar_mse[1],"h15":ar_mse[2],"mean":ar_mse.mean()}]),
    pd.DataFrame([{"model":"PCA","h5":pca_mse[0],"h10":pca_mse[1],"h15":pca_mse[2],"mean":pca_mse.mean()}]),
    ablation_report.drop(columns="best_epoch"),
    pd.DataFrame([{"model":"GNN_seed_average","h5":gnn_seed_mse[0],"h10":gnn_seed_mse[1],"h15":gnn_seed_mse[2],"mean":gnn_seed_mse.mean()}]),
    pd.DataFrame([{"model":"V2_blend","h5":blend_report.blend_MSE.iloc[0],"h10":blend_report.blend_MSE.iloc[1],
                   "h15":blend_report.blend_MSE.iloc[2],"mean":v2_val_mse}])
],ignore_index=True).sort_values("mean").to_csv(OUTPUT_DIR/"v2_experiment_results.csv",index=False)


## 11. Full-data refit

Semua keputusan sudah dikunci oleh validation. Sekarang:

- AR direfit pada full continuous series.
- PCA dan residual Ridge direfit pada full series memakai alpha yang sudah dipilih.
- Arsitektur neural terpilih direfit untuk setiap seed dengan epoch median dari seed search.

Validation tidak dipakai lagi untuk mengubah model setelah cell ini.

In [ ]:
full_times=[valid_times(len(z),HISTORY-1,len(z),FINAL_STRIDE) for z in train_speeds]
if FAST_MODE: full_times=[t[-min(140,len(t)):] for t in full_times]

# Full AR
ar_full=fit_shared_ar(full_times,max_timestamps_per_block=1800)
test_pred_ar=predict_ar_histories(ar_full,test_hist)

# Full PCA
pca_full=fit_road_pca([len(z) for z in train_speeds])
F_full=pca_features_for_times(pca_full,full_times)
F_test=pca_features_for_histories(pca_full,test_hist)
Y_full,B_full=targets_and_ar_for_times(full_times,ar_full)
test_pred_pca_parts=[]
for i,h in enumerate(HORIZONS):
    model=Ridge(alpha=pca_best_alpha[h]).fit(F_full,Y_full[...,i]-B_full[...,i])
    test_pred_pca_parts.append(test_pred_ar[...,i]+model.predict(F_test))
test_pred_pca=np.stack(test_pred_pca_parts,axis=-1).astype(np.float32)
del F_full,Y_full,B_full
gc.collect()

# Full normalization for neural refit
all_rows=np.concatenate([np.asarray(z,dtype=np.float32) for z in train_speeds])
full_mean=all_rows.mean(0).astype(np.float32); full_std=(all_rows.std(0)+3.0).astype(np.float32)
del all_rows; gc.collect()

class TestDataset(Dataset):
    def __len__(self): return len(test_hist)
    def __getitem__(self,i):
        return torch.from_numpy(np.asarray(test_hist[i],dtype=np.float32).T.copy()),torch.from_numpy(test_text_z[i])

test_loader=DataLoader(TestDataset(),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                       pin_memory=torch.cuda.is_available())
test_seed_predictions=[]
for seed in SEEDS:
    print("\n=== FULL REFIT SEED",seed,"===")
    model,log,_,_=train_neural(best_config,full_times,None,full_mean,full_std,seed,refit_epochs)
    model.eval(); parts=[]
    with torch.no_grad():
        for hist,text in test_loader:
            pred,_=model(hist.to(DEVICE),text.to(DEVICE)); parts.append(pred.cpu().numpy())
    test_seed_predictions.append(np.concatenate(parts).astype(np.float32))
    torch.save({"config":asdict(best_config),"seed":seed,"epochs":refit_epochs,
                "state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()}},
               OUTPUT_DIR/f"v2_{best_config.name}_seed{seed}.pt")
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

test_pred_gnn=np.mean(test_seed_predictions,axis=0).astype(np.float32)
W=np.asarray(blend_weights,dtype=np.float32) # horizon, model
test_pred=np.empty_like(test_pred_ar)
for i in range(3):
    test_pred[...,i]=W[i,0]*test_pred_ar[...,i]+W[i,1]*test_pred_pca[...,i]+W[i,2]*test_pred_gnn[...,i]
test_pred=np.clip(test_pred,0,160).astype(np.float32)
print("Test prediction:",test_pred.shape,"range",float(test_pred.min()),float(test_pred.max()),
      "mean",float(test_pred.mean()))


## 12. Submission dan final audit

Internal shape adalah `(sample, road, horizon)`. CSV mengharapkan `(sample, horizon, road)`. Boundary assertions mencegah silent ordering error.

In [ ]:
submission=sample_submission.copy()
flat=test_pred.transpose(0,2,1).reshape(-1)
assert len(flat)==len(submission)
assert submission.id.iloc[0]=="test_00000_h5_r0"
assert submission.id.iloc[N_ROADS]=="test_00000_h10_r0"
assert submission.id.iloc[3*N_ROADS]=="test_00001_h5_r0"
assert submission.id.iloc[-1]=="test_00539_h15_r1259"
submission["speed"]=flat
assert submission.speed.notna().all() and submission.speed.between(0,160).all()
submission_path=OUTPUT_DIR/"submission_v2.csv"
submission.to_csv(submission_path,index=False)

audit=[]
for i,h in enumerate(HORIZONS):
    x=test_pred[...,i]
    audit.append({"horizon":f"h{h}","mean":x.mean(),"std":x.std(),"min":x.min(),"max":x.max(),
                  "zeros":int(np.sum(x==0)),"mean_abs_vs_mean3":np.mean(np.abs(x-test_hist[:,-3:].mean(1)))})
display(pd.DataFrame(audit).round(5))
print("Saved:",submission_path,"rows",len(submission),"NaN",int(submission.speed.isna().sum()))
display(pd.concat([submission.head(3),submission.tail(3)]))


## 13. Cara membaca hasil dan next decision

1. Lihat `ablation_report`: ini menjawab apakah graph dan text benar-benar membantu.
2. Lihat `seed_report`: seed yang sangat buruk menunjukkan variance/instability.
3. Lihat `blend_report`: bobot nol berarti model tersebut tidak menambah informasi untuk horizon terkait.
4. Bandingkan `V2 validation mean` dengan referensi V1 27.8935.
5. Submit V2 sebagai eksperimen terpisah; jangan menghapus V1 sebelum leaderboard dibandingkan.

Jika V2 validation belum menang:

- submission tetap valid, tetapi jangan mengklaim improvement;
- gunakan hasil ablation untuk menghapus komponen yang tidak membantu;
- prioritaskan rolling validation sebelum memperbesar arsitektur.

Jika V2 menang tetapi leaderboard turun, kemungkinan temporal validation belum merepresentasikan test distribution. Langkah berikutnya adalah leave-one-block-out/rolling folds dan rank/weighted blending berdasarkan stabilitas lintas fold.